In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from keras.datasets import imdb
from keras.preprocessing.sequence import pad_sequences
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
VOCAB_SIZE = 10000
MAX_SEQ_LENGTH = 256
BATCH_SIZE = 512

(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words= VOCAB_SIZE)


In [ ]:
x_train = pad_sequences(train_data, 
                        padding= 'post',
                        value = 0)

x_test = pad_sequences(test_data,
                       padding='post',
                       value = 0)

y_train = np.array(train_labels)
y_test = np.array(test_labels)

x_train_tensor = torch.tensor(x_train, dtype= torch.long)
y_train_tensor = torch.tensor(y_train, dtype= torch.float32)

x_test_tensor = torch.tensor(x_test, dtype= torch.long)
y_test_tensor = torch.tensor(y_test, dtype= torch.float32)

val_size = int(0.2*len(x_train_tensor))

x_val_tensor, x_train_tensor = x_train_tensor[:val_size], x_train_tensor[val_size:]
y_val_tensor, y_train_tensor = y_train_tensor[:val_size], y_train_tensor[val_size:]

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size= BATCH_SIZE, shuffle= True)
val_loader = DataLoader(val_dataset, batch_size= BATCH_SIZE, shuffle= False)
test_loader = DataLoader(test_dataset, batch_size= BATCH_SIZE, shuffle= False)

In [ ]:
class SentimentGRU(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings= vocab_size,
                                      embedding_dim= embedding_size,
                                      padding_idx= 0)
        
        self.gru = nn.GRU(input_size= embedding_size,
                          batch_first= True,
                          hidden_size= hidden_size)
        
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)

        output, hn = self.gru(embedded)
        
        last_hidden = output[:, -1, :]

        out = self.fc(last_hidden)
        out = self.sigmoid(out)

        return out.squeeze()

embedding_dim = 16
rnn_units = 32

model = SentimentGRU(VOCAB_SIZE, embedding_dim, rnn_units).to(device)


In [ ]:
EPOCHS = 10
optimizer = optim.Adam(model.parameters(), lr = 0.001)
criterion = nn.BCELoss()

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

def calculate_accuracy(preds, labels):
    rounded_preds = torch.round(preds)

    correct = (rounded_preds == labels).float()

    acc = correct.sum()/len(labels)

    return acc


print("Starting train ....")
for epoch in range(EPOCHS):

    model.train()

    epoch_loss = 0
    epoch_acc = 0
    # train
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()

        predictions = model(batch_x)

        loss = criterion(predictions, batch_y)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += calculate_accuracy(predictions, batch_y).item()

    train_loss = epoch_loss/ len(train_loader)
    train_acc = epoch_acc/ len(train_loader)

    # check val

    model.eval()
    val_loss = 0
    val_acc = 0

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            predictions = model(batch_x)

            loss = criterion(predictions, batch_y)

            val_loss += loss.item()
            val_acc += calculate_accuracy(predictions, batch_y).item()
            
    val_loss = val_loss / len(val_loader)
    val_acc = val_acc / len(val_loader)
    
    # Lưu vào lịch sử
    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

print("Huấn luyện hoàn tất!")




In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

epochs_range = list(range(1, EPOCHS + 1))

fig = make_subplots(rows=1, cols=2, subplot_titles=("Training and Validation Loss", "Training and Validation Accuracy"))

# Vẽ Loss
fig.add_trace(go.Scatter(x=epochs_range, y=history['loss'], name='Training Loss', mode='lines+markers', line=dict(color='#4263eb')), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_range, y=history['val_loss'], name='Validation Loss', mode='lines+markers', line=dict(color='#f76707')), row=1, col=1)

# Vẽ Accuracy
fig.add_trace(go.Scatter(x=epochs_range, y=history['accuracy'], name='Training Accuracy', mode='lines+markers', line=dict(color='#12b886')), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs_range, y=history['val_accuracy'], name='Validation Accuracy', mode='lines+markers', line=dict(color='#ae3ec9')), row=1, col=2)

fig.update_layout(height=400, width=900, title_text="PyTorch Model Training History")
fig.update_xaxes(title_text="Epochs", row=1, col=1)
fig.update_xaxes(title_text="Epochs", row=1, col=2)
fig.update_yaxes(title_text="Loss", row=1, col=1)
fig.update_yaxes(title_text="Accuracy", row=1, col=2)

fig.show()